<center><h1>Verma_Karan_HW3</h1></center>

**Name:** Karan Verma  
**GitHub Username:** *kv304*  
**USC ID:** *5639810577*

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports used in this notebook.

In [4]:
from pathlib import Path
from zipfile import ZipFile
import re

import numpy as np
import pandas as pd
from IPython.display import display

The AReM data have seven activity classes. Each file is one activity instance and normally has 480 rows for six sensor measurements. The ZIP file is already in the homework folder, so I only extract it when the `AReM` directory is missing.

In [5]:
data_dir = Path("hw3_4_data")
zip_file = data_dir / "AReM.zip"
arem_dir = data_dir / "AReM"

if not arem_dir.exists():
    with ZipFile(zip_file, "r") as archive:
        archive.extractall(data_dir)

print("AReM folder:", arem_dir.resolve())
print("Folder exists:", arem_dir.exists())

AReM folder: /Users/karanverma/Documents/USC_classes/DSCI552/hw3/hw3_4_data/AReM
Folder exists: True


In [6]:
sensor_cols = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23",
]

all_cols = ["time"] + sensor_cols
def load_sensor_file(file_path):
    rows = []

    with open(file_path, "r", encoding="utf-8-sig") as file:
        for line in file:
            cleaned_line = line.strip().rstrip(",")

            # Skip headers and empty lines.
            if not cleaned_line or cleaned_line.startswith("#"):
                continue

            # The split also handles the one space-delimited file.
            values = re.split(r"[,\s]+", cleaned_line)

            if len(values) != 7:
                raise ValueError(
                    f"Unexpected row in {file_path.name}: {cleaned_line}"
                )

            rows.append([float(value) for value in values])

    return pd.DataFrame(rows, columns=all_cols)


### (b) Test and Train Data

I used the split specified in the assignment. For `bending1` and `bending2`, datasets 1 and 2 are test cases. For the other five activities, datasets 1--3 are test cases. This leaves 69 training instances and 19 test instances.

In [7]:
activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking",
]


def parse_dataset_id(file_path):
    match = re.search(r"dataset(\d+)", file_path.stem)
    return int(match.group(1))


records = []

for activity in activities:
    activity_folder = arem_dir / activity

    files = sorted(
        activity_folder.glob("dataset*.csv"),
        key=parse_dataset_id,
    )

    for file_path in files:
        dataset_number = parse_dataset_id(file_path)

        if activity in ["bending1", "bending2"]:
            is_test = dataset_number in [1, 2]
        else:
            is_test = dataset_number in [1, 2, 3]

        sensor_data = load_sensor_file(file_path)

        records.append(
            {
                "activity": activity,
                "dataset": dataset_number,
                "split": "test" if is_test else "train",
                "observations": len(sensor_data),
                "file_path": file_path,
            }
        )

manifest = pd.DataFrame(records)

In [8]:
manifest.head()

,activity,dataset,split,observations,file_path
0,bending1,1,test,480,hw3_4_data/AReM/bending1/dataset1.csv
1,bending1,2,test,480,hw3_4_data/AReM/bending1/dataset2.csv
2,bending1,3,train,480,hw3_4_data/AReM/bending1/dataset3.csv
3,bending1,4,train,480,hw3_4_data/AReM/bending1/dataset4.csv
4,bending1,5,train,480,hw3_4_data/AReM/bending1/dataset5.csv


In [9]:
split_counts = (
    manifest
    .groupby(["activity", "split"])
    .size()
    .unstack(fill_value=0)
)

display(split_counts)

print("Total instances:", len(manifest))
print("Training instances:", (manifest["split"] == "train").sum())
print("Test instances:", (manifest["split"] == "test").sum())

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


Total instances: 88
Training instances: 69
Test instances: 19


In [10]:
row_counts = manifest["observations"].value_counts().sort_index()

row_counts

observations
479     1
480    87
Name: count, dtype: int64

In [11]:

manifest.loc[
    manifest["observations"] != 480,
    ["activity", "dataset", "observations"]
]


,activity,dataset,observations
50,sitting,8,479


### (c) Feature Extraction


#### i. Research

Time-domain features summarize a signal directly from its observed values. Examples include the minimum, maximum, mean, median, variance, standard deviation, quartiles, interquartile range, RMS, skewness, kurtosis, and zero-crossing rate. For this homework I calculate the seven requested statistics: minimum, maximum, mean, median, standard deviation, first quartile, and third quartile.

#### ii. Extraction

There are six sensor series and seven statistics per series, giving $6 \times 7 = 42$ features for each instance. The suffix identifies the sensor series; for example, `mean1` is the mean of the first series and `q3_6` is the third quartile of the sixth.

In [12]:
def extract_features(sensor_data):
    feature_values = {}

    for series_number, signal in enumerate(sensor_cols, start=1):
        measurements = sensor_data[signal]

        feature_values[f"min{series_number}"] = measurements.min()
        feature_values[f"max{series_number}"] = measurements.max()
        feature_values[f"mean{series_number}"] = measurements.mean()
        feature_values[f"median{series_number}"] = measurements.median()
        feature_values[f"std{series_number}"] = measurements.std(ddof=1)
        feature_values[f"q1_{series_number}"] = measurements.quantile(0.25)
        feature_values[f"q3_{series_number}"] = measurements.quantile(0.75)

    return feature_values

In [13]:
feature_records = []

for instance in manifest.itertuples(index=False):
    sensor_data = load_sensor_file(instance.file_path)
    extracted_features = extract_features(sensor_data)

    feature_records.append(
        {
            "activity": instance.activity,
            "dataset": instance.dataset,
            "split": instance.split,
            **extracted_features,
        }
    )

features = pd.DataFrame(feature_records)

display(features.head())
print("Feature table shape:", features.shape)

,activity,dataset,split,min1,max1,mean1,median1,std1,q1_1,q3_1,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
0,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


Feature table shape: (88, 45)


In [14]:
train_df = features[
    features["split"] == "train"
].reset_index(drop=True)

test_df = features[
    features["split"] == "test"
].reset_index(drop=True)

print("Training feature table:", train_df.shape)
print("Test feature table:", test_df.shape)


Training feature table: (69, 45)
Test feature table: (19, 45)


In [15]:
id_cols = ["activity", "dataset", "split"]

feature_cols = [
    column
    for column in features.columns
    if column not in id_cols
]

print("Number of extracted features:", len(feature_cols))
print(
    "Number of missing feature values:",
    features[feature_cols].isna().sum().sum()
)

assert len(feature_cols) == 42
assert features[feature_cols].isna().sum().sum() == 0


Number of extracted features: 42
Number of missing feature values: 0


In [16]:
display(
    features
    .sort_values(["activity", "dataset"])
    .reset_index(drop=True)
)


,activity,dataset,split,min1,max1,mean1,median1,std1,q1_1,q3_1,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
0,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,walking,11,train,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,walking,12,train,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,walking,13,train,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,walking,14,train,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


#### iii. Standard Deviation

I first compute the sample standard deviation of each of the 42 features across all 88 instances. To obtain 90% bootstrap intervals, I resample the 88 rows with replacement 10,000 times and recompute all 42 standard deviations for every resample. The 5th and 95th percentiles of those bootstrap values are the interval endpoints.

In [17]:
X_features = features[feature_cols].to_numpy()

observed_sd = X_features.std(axis=0, ddof=1)

sd_summary = pd.Series(
    observed_sd,
    index=feature_cols,
    name="Estimated SD",
)

sd_summary.to_frame().round(4)

,Estimated SD
min1,9.5700
max1,4.3944
mean1,5.3357
median1,5.4401
std1,1.7722
q1_1,6.1536
q3_1,5.1389
min2,0.0000
max2,5.0627
mean2,1.5742


In [18]:
rng = np.random.default_rng(552)

n_boot = 10000
n_instances = len(X_features)
n_features = len(feature_cols)

boot_sd = np.empty(
    (n_boot, n_features)
)

for b in range(n_boot):
    row_idx = rng.integers(
        low=0,
        high=n_instances,
        size=n_instances,
    )

    boot_sample = X_features[row_idx]

    boot_sd[b] = boot_sample.std(
        axis=0,
        ddof=1,
    )

In [19]:
ci_low = np.percentile(
    boot_sd,
    5,
    axis=0,
)

ci_high = np.percentile(
    boot_sd,
    95,
    axis=0,
)

bootstrap_summary = pd.DataFrame(
    {
        "Estimated SD": observed_sd,
        "90% CI Lower": ci_low,
        "90% CI Upper": ci_high,
    },
    index=feature_cols,
)

bootstrap_summary.index.name = "Feature"

display(bootstrap_summary.T.round(4))

Feature,min1,max1,mean1,median1,std1,q1_1,q3_1,min2,max2,mean2,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
Estimated SD,9.5700,4.3944,5.3357,5.4401,1.7722,6.1536,5.1389,0.0,5.0627,1.5742,...,1.0249,6.0965,5.5317,0.0458,2.5189,1.1548,1.0865,0.5176,0.7586,1.5236
90% CI Lower,8.2622,3.3280,4.7040,4.7903,1.5722,5.5816,4.3319,0.0,4.6228,1.3941,...,0.8132,4.7701,4.3398,0.0000,2.2508,1.0601,0.9925,0.4781,0.6882,1.3985
90% CI Upper,10.7614,5.2849,5.8864,6.0043,1.9500,6.6524,5.8409,0.0,5.4022,1.7068,...,1.2158,7.2176,6.5283,0.0785,2.7639,1.2163,1.1487,0.5455,0.8087,1.6015


In [20]:
bootstrap_summary["CI Width"] = (
    bootstrap_summary["90% CI Upper"]
    - bootstrap_summary["90% CI Lower"]
)

display(
    bootstrap_summary
    .sort_values("CI Width", ascending=False)
    .round(4)
)

,Estimated SD,90% CI Lower,90% CI Upper,CI Width
Feature,,,,
min5,6.1240,4.4033,7.5110,3.1077
min1,9.5700,8.2622,10.7614,2.4992
q1_5,6.0965,4.7701,7.2176,2.4476
median5,5.8138,4.4902,6.9103,2.4201
mean5,5.6756,4.4016,6.7385,2.3369
q3_5,5.5317,4.3398,6.5283,2.1885
max1,4.3944,3.3280,5.2849,1.9569
max5,5.7412,4.7406,6.5541,1.8134
q3_1,5.1389,4.3319,5.8409,1.5089


#### iv. Select Features

I would keep **minimum, mean, and maximum** as a compact three-feature summary. The mean represents the overall signal level, while the minimum and maximum retain information about the observed extremes. Quartiles and the median would be more robust to isolated extreme values, and standard deviation could help separate steady activities from more variable ones. Still, min/mean/max give a simple and interpretable starting point.

## 2. ISLR 3.7.4

The two fitted models are

\\[Y = \\beta\_0 + \\beta\_1X + \\epsilon\\]

and

\\[Y = \\beta\_0 + \\beta\_1X + \\beta\_2X^2 + \\beta\_3X^3 + \\epsilon\\]

The linear model is nested inside the cubic model because setting \(\beta_2=\beta_3=0\) gives the linear form.

### (a) Linear Train

The cubic fit must have training RSS less than or equal to the linear fit. It includes every term in the linear model, so least squares can set the two additional coefficients to zero if they do not help. Otherwise, the extra terms can reduce training RSS.

### (b) Linear Test

I expect the linear model to have lower test RSS because the true relationship is stated to be linear. The quadratic and cubic terms add variance without correcting any model bias, so they can fit training noise and generalize worse.

### (c) Not Linear Train

The cubic model still has training RSS less than or equal to the linear model. This follows from the models being nested and does not depend on the actual shape of the relationship.

### (d) Not Linear Testing

There is not enough information to rank their test RSS. If the unknown relationship is close to cubic, the cubic model may reduce bias. If it has a different shape or the sample is noisy, the extra terms may instead increase variance. The result therefore depends on the function, noise level, and sample size.

In [21]:
checks = {
    "Total instances": len(manifest),
    "Training instances": len(train_df),
    "Test instances": len(test_df),
    "Extracted features": len(feature_cols),
    "Missing feature values": int(
        features[feature_cols].isna().sum().sum()
    ),
    "Bootstrap repetitions": n_boot,
}

validation_table = pd.DataFrame.from_dict(
    checks,
    orient="index",
    columns=["Value"],
)

display(validation_table)

assert checks["Total instances"] == 88
assert checks["Training instances"] == 69
assert checks["Test instances"] == 19
assert checks["Extracted features"] == 42
assert checks["Missing feature values"] == 0
assert bootstrap_summary.shape == (42, 4)

print("All validation checks passed.")


,Value
Total instances,88
Training instances,69
Test instances,19
Extracted features,42
Missing feature values,0
Bootstrap repetitions,10000


All validation checks passed.


## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 